In [ ]:
#!uv pip install -e /home/tomasruiz/code/TransformerLens
# !uv pip install sae-lens

import torch
from transformers import AutoTokenizer
from transformer_lens import HookedTransformer
from sae_library import test_heavens_and_earth_prompt

from sae_lens import SAE

torch.set_grad_enabled(False)  # avoid blowing up mem
device = "cuda"
model_id = "google/gemma-2-2b"

# Gemma2 LM

In [ ]:
model = HookedTransformer.from_pretrained(model_id, torch_dtype=torch.bfloat16)
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
from transformers import AutoModelForCausalLM

model2 = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
).eval()
tokenizer2 = AutoTokenizer.from_pretrained(model_id)

In [ ]:
from sae_library import GemmaWrapper

model3 = GemmaWrapper(model2, tokenizer2, device)
print(model3.to_str_tokens("hello world", prepend_bos=True))
print(model3.to_str_tokens("hello world", prepend_bos=False))

In [ ]:
test_heavens_and_earth_prompt(model)

In [ ]:
test_heavens_and_earth_prompt(model3)

In [2]:
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id="layer_20/width_16k/canonical",
    device=device,
)

In [ ]:
text = "Would you be able to travel through time using a wormhole?"

batch_tokens = tokenizer.encode(text, return_tensors="pt").to(device)
print(tokenizer.batch_decode(batch_tokens))  # includes BOS token
_, cache = model.run_with_cache(batch_tokens, prepend_bos=False)

In [ ]:
acts = cache[sae.cfg.hook_name]
del cache
features = sae.encode(acts[:, 1:, :])  # exclude the BOS token

print(acts.shape)
print(features.shape)


In [ ]:
(features > 0).sum(dim=2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.Series(features.flatten().cpu().numpy()).describe()

In [ ]:
# top k=1 feature for each token
k = 1
top_k_features_per_token = (
    features.abs().argsort(dim=2, descending=True)[:, :, :k].long().flatten()
)
top_k_features_per_token


In [ ]:
# top k=10 features of the last token
k = 10
top_k_features_last_token = (
    features[:, -1, :].abs().argsort(descending=True)[0, :k].long()
)
top_k_features_last_token


In [ ]:
import pandas as pd

# increase display width
pd.options.display.max_colwidth = 1000

explanations = pd.read_parquet("features/explanations.parquet")
explanations = explanations.query("explanationModelName == 'gpt-4o-mini'")
explanations["feature_id"] = explanations["index"].astype(int)
explanations.set_index("feature_id", inplace=True)
top_k_explanations = explanations.loc[
    top_k_features_per_token.cpu().numpy()
].reset_index()
top_k_explanations = top_k_explanations.assign(
    token=tokenizer.convert_ids_to_tokens(batch_tokens[0, 1:])
)
top_k_explanations[["token", "description", "feature_id"]]


# PaliGemma2

from PIL import Image

img = Image.open("pic.jpg")
text = "describe en"
inputs = (
    pgprocessor(text="<image> " + text, images=img, return_tensors="pt")
    .to(torch.bfloat16)
    .to(device)
)
input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = pgmodel.generate(**inputs, max_new_tokens=100, do_sample=False)
generation = generation[0][input_len:]
decoded: str = pgprocessor.decode(generation, skip_special_tokens=True)
decoded
